# Claude agentic tools on Amazon Bedrock — computer use, memory, compaction

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

The Anthropic-defined tool families: **computer use** (GUI automation), **bash**
and **text editor**, the **memory** tool, and **context management** (compaction).
These are what let a Claude agent run for a long time without drowning in its own
context.

**Prerequisites:** `01-messages-api-core.ipynb` (Messages API basics) and
`02-thinking-tools-and-caching.ipynb` (thinking, tools, caching).

## ⚠️ Beta services and real risk
Computer use is a **Beta Service** under the AWS Service Terms. It carries risks
that ordinary chat APIs do not:

- Run it in a **dedicated VM or container with minimal privileges**.
- Do not give it access to sensitive accounts or data.
- Restrict its network reach to the domains it needs.
- Keep a **human in the loop** for anything consequential.

Anything the model can see can influence it — this is a prompt-injection surface.
**This notebook never executes a real GUI action.** It shows the protocol and
simulates the environment, which is what you want when learning it.

## Region
This notebook pins `us-east-1`. Model availability differs by Region and changes over
time, so verify with `GET /v1/models` for whichever Region you plan to use.

## Self-contained, but see also
- `01-messages-api-core.ipynb` · `02-thinking-tools-and-caching.ipynb`
- **Auth, the three URL paths** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs anthropic, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `converse` | one Converse call; returns `(text, response)` and **never raises** on a service error |
| `err` | pulls the human-readable message out of an error body, redacted |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |
| `converse_text` | concatenates the text blocks of a Converse response — safer than `content[0]` |
| `converse_tool_uses` | the `toolUse` blocks from a Converse response |
| `resolve_runtime_id` | turns a model ID into the form Converse will accept, adding the `us.` profile prefix when one is required |
| `runtime_client` | a boto3 `bedrock-runtime` client (Converse, InvokeModel) |
| `converse_reasoning` | the reasoning trace from a Converse response, or `""` |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import json
import sys

sys.path.insert(0, "../_shared")
from bedrock import converse, err, post, slide_jpeg

REGION = "us-east-1"

OPUS5 = "anthropic.claude-opus-5"
OPUS48 = "anthropic.claude-opus-4-8"
SONNET5 = "anthropic.claude-sonnet-5"
HAIKU45 = "anthropic.claude-haiku-4-5"

PREFIX = "/anthropic/v1"
AV = {"anthropic-version": "2023-06-01"}

# Beta features are opted into with the anthropic-beta HEADER on mantle.
# (On bedrock-runtime the equivalent goes in the body as "anthropic_beta".)
COMPUTER_USE_BETA = "computer-use-2025-11-24"
CONTEXT_MGMT_BETA = "context-management-2025-06-27"


def claude_text(payload: dict) -> str:
    return "".join(
        b.get("text", "") for b in payload.get("content", []) if b.get("type") == "text"
    )


def tool_uses(payload: dict) -> list:
    return [b for b in payload.get("content", []) if b.get("type") == "tool_use"]


print("endpoint:", f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}")

endpoint: https://bedrock-mantle.us-east-1.api.aws/anthropic/v1


### Which endpoint, and the model ID for each

AWS recommends `bedrock-runtime` for new applications, and since August 2026 it
serves the OpenAI- and Anthropic-compatible APIs as well as Converse. So before
the first call, the question is which endpoint you want — and that has a
complication worth knowing about:

**the same model often carries a different ID on each endpoint.** Send a
`bedrock-mantle` ID to `bedrock-runtime` and you get *"The provided model
identifier is invalid"*, which reads like a missing model rather than a missing
translation.

The cell below asks both catalogues rather than stating an answer that will age.
`runtime_id_for()` returns `None` when a model is genuinely not on
`bedrock-runtime`, which is the honest signal for "you need mantle for this one".

In [2]:
from bedrock import endpoints_for, runtime_id_for

COVERED = [
    "anthropic.claude-haiku-4-5",
    "anthropic.claude-opus-4-8",
    "anthropic.claude-opus-5",
    "anthropic.claude-sonnet-5",
]

print(f"{'model (as named on mantle)':38} {'on runtime as':40} endpoints")
print("-" * 96)
mantle_only = []
for model_id in COVERED:
    runtime_id = runtime_id_for(model_id, REGION)
    where = endpoints_for(model_id, REGION)
    label = ", ".join(name for name, present in where.items() if present) or "neither"
    if runtime_id is None:
        mantle_only.append(model_id)
    print(f"{model_id:38} {(runtime_id or '-- not on runtime --'):40} {label}")

renamed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m
]
# Cross-check the two helpers against each other. A row that prints a runtime id
# next to "mantle" only is self-contradictory, and it happened: endpoints_for()
# compared against a version-stripped catalogue key while runtime_id_for() used the
# full id, so gpt-oss showed a runtime id and "mantle". Neither helper complained.
contradictions = [
    m for m in COVERED
    if (runtime_id_for(m, REGION) is not None)
    != endpoints_for(m, REGION)["runtime"]
]
print()
if contradictions:
    print(f"!! runtime_id_for() and endpoints_for() DISAGREE for {contradictions}.")
    print("   One of them is wrong; do not trust the table above until they agree.")
print(f"=> {len(COVERED) - len(mantle_only)}/{len(COVERED)} of these are on "
      f"bedrock-runtime; {len(renamed)} under a different id.")
if mantle_only:
    print(f"   bedrock-mantle only: {mantle_only}")
    print("   For those, this notebook's endpoint is the only one that serves them.")
else:
    print("   Every model here is on both endpoints. This notebook shows the")
    print("   bedrock-mantle calls; the ids above are what you send to switch.")
print("   Region matters too: a model absent here can be present elsewhere, so")
print("   re-run this in the Region you intend to deploy in.")

model (as named on mantle)             on runtime as                            endpoints
------------------------------------------------------------------------------------------------


anthropic.claude-haiku-4-5             us.anthropic.claude-haiku-4-5-20251001-v1:0 mantle, runtime


anthropic.claude-opus-4-8              us.anthropic.claude-opus-4-8             mantle, runtime


anthropic.claude-opus-5                us.anthropic.claude-opus-5               mantle, runtime


anthropic.claude-sonnet-5              us.anthropic.claude-sonnet-5             mantle, runtime



=> 4/4 of these are on bedrock-runtime; 4 under a different id.
   Every model here is on both endpoints. This notebook shows the
   bedrock-mantle calls; the ids above are what you send to switch.
   Region matters too: a model absent here can be present elsewhere, so
   re-run this in the Region you intend to deploy in.


## 1. How beta tools are enabled on mantle

Two things have to line up: the **beta header** and a **tool `type` that matches
that beta version**. The failure is one-sided, which is worth seeing — a tool type
without its header is a 400, but the header on its own is harmless.

In [3]:
computer_tool = {
    "type": "computer_20251124",  # must match the beta version below
    "name": "computer",
    "display_width_px": 1024,
    "display_height_px": 768,
}

probes = [
    ("correct: header + tool", {COMPUTER_USE_BETA: True}, [computer_tool]),
    ("tool without beta header", {}, [computer_tool]),
    ("header without the tool", {COMPUTER_USE_BETA: True}, None),
]
for label, beta, tools in probes:
    headers = dict(AV)
    if beta:
        headers["anthropic-beta"] = COMPUTER_USE_BETA
    body = {
        "model": OPUS48,
        "max_tokens": 200,
        "messages": [{"role": "user", "content": "Take a screenshot."}],
    }
    if tools:
        body["tools"] = tools
    code, data = post(f"{PREFIX}/messages", body, region=REGION, headers=headers)
    print(f"  {label:26} -> HTTP {code} {'' if code == 200 else err(data)[:70]}")

print()
print("=> The tool type without its beta header is rejected, and the message names")
print("   the tag it could not match. The header without a matching tool is simply")
print("   ignored, so an unused anthropic-beta value costs nothing.")

  correct: header + tool     -> HTTP 200 


  tool without beta header   -> HTTP 400 tools.0: Input tag 'computer_20251124' found using 'type' does not mat


  header without the tool    -> HTTP 200 

=> The tool type without its beta header is rejected, and the message names
   the tag it could not match. The header without a matching tool is simply
   ignored, so an unused anthropic-beta value costs nothing.


## 2. Which models support computer use?

Support varies by model. Probe before you build.

One aside on the output: the refusal quotes an internal service-side alias for the
model rather than the ID you sent. That is a reminder that error *strings* are not a
stable contract even when the behaviour they describe is — match on status codes and
documented error types, not on message text.

In [4]:
print(f"{'model':32} {'computer use':>14}")
print("-" * 50)
for model in (OPUS5, OPUS48, SONNET5, HAIKU45):
    code, data = post(
        f"{PREFIX}/messages",
        {
            "model": model,
            "max_tokens": 200,
            "tools": [computer_tool],
            "messages": [{"role": "user", "content": "Take a screenshot."}],
        },
        region=REGION,
        headers={**AV, "anthropic-beta": COMPUTER_USE_BETA},
    )
    verdict = "supported" if code == 200 else f"{code}"
    print(f"{model:32} {verdict:>14}")
    if code != 200:
        print(f"      {err(data)[:80]}")

model                              computer use
--------------------------------------------------


anthropic.claude-opus-5                     400
      'claude-honey' does not support tool types: computer_20251124. Did you mean one 


anthropic.claude-opus-4-8             supported


anthropic.claude-sonnet-5             supported


anthropic.claude-haiku-4-5                  400
      'claude-haiku-4-5-20251001' does not support tool types: computer_20251124. Did 


## 3. The computer-use protocol

The model returns a `tool_use` block describing an **action** — `screenshot`,
`left_click`, `type`, `scroll`, `key`. Your application performs it and returns a
`tool_result`. Nothing happens unless your code makes it happen, which is where
your safety controls belong.

In [5]:
code, data = post(
    f"{PREFIX}/messages",
    {
        "model": OPUS48,
        "max_tokens": 600,
        "tools": [computer_tool],
        "messages": [
            {
                "role": "user",
                "content": (
                    "Take a screenshot of the desktop so we can " "see what's open."
                ),
            }
        ],
    },
    region=REGION,
    headers={**AV, "anthropic-beta": COMPUTER_USE_BETA},
)
print("HTTP", code, "| stop_reason:", data.get("stop_reason"))
print("content block types:", [b.get("type") for b in data.get("content", [])])
for block in tool_uses(data):
    print(f"\ntool: {block['name']}")
    print("input:", json.dumps(block["input"], indent=2))

HTTP 200 | stop_reason: tool_use
content block types: ['text', 'tool_use']

tool: computer
input: {
  "action": "screenshot"
}


## 4. A simulated computer-use loop

We return **synthetic** screenshots and results. This exercises the full protocol
— including the human-approval gate — without automating anything real.

In [6]:
import base64

# A real screenshot: one frame of a public AWS talk. Standing in for whatever
# the agent would actually be looking at. See ../_shared/bedrock.py.
SCREENSHOT = base64.b64encode(slide_jpeg()).decode()

# Actions we are willing to perform without asking a human first.

The approval gate. A computer-use agent can click anything on screen, so decide
up front which actions run unattended and which need a human (OWASP LLM06,
Excessive Agency).

In [7]:
AUTO_APPROVED = {"screenshot", "cursor_position"}


def perform_action(action: dict) -> dict:
    """Simulate a GUI action. A real implementation would drive a sandboxed VM."""
    kind = action.get("action")
    if kind not in AUTO_APPROVED:
        # THE HUMAN-IN-THE-LOOP GATE. In a real system, block here for approval.
        return {
            "refused": True,
            "reason": f"action '{kind}' requires human approval in this environment",
        }
    if kind == "screenshot":
        return {
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": "image/jpeg",
                "data": SCREENSHOT,
            },
        }
    return {"ok": True, "action": kind}

The loop, bounded by `max_rounds`.

In [8]:
def computer_use_loop(task: str, model: str = OPUS48, max_rounds: int = 4) -> list:
    conversation = [{"role": "user", "content": task}]
    trace = []
    for round_no in range(max_rounds):
        code, data = post(
            f"{PREFIX}/messages",
            {
                "model": model,
                "max_tokens": 900,
                "tools": [computer_tool],
                "messages": conversation,
            },
            region=REGION,
            headers={**AV, "anthropic-beta": COMPUTER_USE_BETA},
        )
        if code != 200:
            trace.append({"round": round_no + 1, "error": err(data)[:100]})
            break
        blocks = tool_uses(data)
        if not blocks:
            trace.append({"round": round_no + 1, "final": claude_text(data)[:200]})
            break
        conversation.append({"role": "assistant", "content": data["content"]})
        results = []
        for block in blocks:
            outcome = perform_action(block["input"])
            trace.append(
                {
                    "round": round_no + 1,
                    "action": block["input"].get("action"),
                    "outcome": (
                        "screenshot returned"
                        if outcome.get("type") == "image"
                        else json.dumps(outcome)[:80]
                    ),
                }
            )
            if outcome.get("type") == "image":
                content = [outcome]  # image block back to Claude
            else:
                content = json.dumps(outcome)
            results.append(
                {"type": "tool_result", "tool_use_id": block["id"], "content": content}
            )
        conversation.append({"role": "user", "content": results})
    return trace

Run it and watch each action pass through the gate.

In [9]:
for step in computer_use_loop("Take a screenshot, then describe what you can see."):
    print(json.dumps(step))

{"round": 1, "action": "screenshot", "outcome": "screenshot returned"}
{"round": 2, "final": "Here's what I can see on the screen:\n\nThis is a **presentation slide** with a dark background, titled **\"Ingestion from database.\"** It appears to be an AWS architecture/technical diagram explaining d"}


Notice the gate: any action outside `AUTO_APPROVED` comes back as a refusal, and
the model has to react to that. That is the shape of a safe integration — the
model proposes, your code decides.

## 4b. A second generation of the same tools, and the action moved

Everything above uses the dated tool types from §1, which are beta-gated. The same
families are also served as **toolsets** — `computer_toolset_20260801`,
`browser_toolset_20260801` — and those are not beta: the cells in this section send no
`anthropic-beta` header anywhere, which is the point of them.

Three things differ from the shape §3 and §4 use, and the third is the one that will
break your code:

- a toolset entry carries **no `name`**; the members are fixed by the dated type;
- support is per model, and a model that does not take the type refuses with its own
  list of tool types. That list is not the one a missing beta header produces, and the
  last cell of this section measures the difference, so check which refusal you have
  before reading it as a capability that went away;
- the `tool_use` block names the **member** rather than the family, leaves its
  arguments in `input`, and adds a `toolset_name` field. The members are whatever the
  model reaches for; read the names the cells below print.

So §4's `perform_action()` reads `input["action"]`, and under a toolset there is no such
key. This section sits after §4 so the second cell can run §4's own gate against both
shapes rather than describing what would happen to it.


In [10]:
import re

from bedrock import runtime_post

# No anthropic-beta header anywhere in this section. Sending one would work, and would
# hide the fact that these tool types do not need it.
SCREENSHOT_ASK = [{"role": "user", "content": "Take a screenshot of the desktop."}]
BROWSER_ASK = [
    {"role": "user", "content": "Open https://example.com and take a screenshot "
                                "of the page."}
]
# One ask per family. The desktop ask sent to the browser toolset comes back as prose
# with no tool_use, which reads like missing support when it is only a mismatched
# request, and it is why the last cell of this section could not see across toolsets
# until each got its own.
ASKS = {"computer_toolset_20260801": SCREENSHOT_ASK,
        "browser_toolset_20260801": BROWSER_ASK}
TOOLSETS = tuple(ASKS)


def toolset_call(model: str, tool_type: str, endpoint: str) -> tuple[int, dict]:
    """One toolset request against 'mantle' or 'runtime'.

    Never raises on HTTP errors: a 4xx comes back as (code, error_body), which is
    the point of the table below.
    """
    body = {
        "model": model,
        "max_tokens": 300,
        "messages": ASKS[tool_type],
        "tools": [{"type": tool_type}],  # no "name" -- see the last cell of this section
    }
    if endpoint == "mantle":
        return post(f"{PREFIX}/messages", body, region=REGION, headers=dict(AV))
    runtime_id = runtime_id_for(model, REGION)
    if runtime_id is None:
        return -1, {"error": {"message": "not on bedrock-runtime"}}
    body["model"] = runtime_id
    # bedrock-runtime wants the version in the BODY and wants it on every request, not
    # only when tools are present. The tool-type validator runs first, so a missing
    # anthropic_version hides behind a tool error and looks like a tool problem.
    body["anthropic_version"] = "bedrock-2023-05-31"
    return runtime_post(f"{PREFIX}/messages", body, region=REGION)


# A refusal from the per-model validator quotes that model's own tool types.
OFFERED = re.compile(r"Did you mean one of ([^?]+)\?")

status: dict[tuple[str, str, str], int] = {}
offered: dict[tuple[str, str], list[str]] = {}
print(f"{'model':28} {'tool type':28} {'mantle':>7} {'runtime':>8}")
print("-" * 75)
for model in COVERED:
    for tool_type in TOOLSETS:
        codes = []
        for endpoint in ("mantle", "runtime"):
            code, data = toolset_call(model, tool_type, endpoint)
            status[(model, tool_type, endpoint)] = code
            codes.append(code)
            if code != 200:
                # err() truncates at 160 chars by default and the per-model list sits
                # past that, so ask for enough of the message to match against.
                hit = OFFERED.search(err(data, limit=600))
                if hit:
                    offered[(model, endpoint)] = [
                        t.strip() for t in hit.group(1).split(",")
                    ]
        print(f"{model:28} {tool_type:28} {codes[0]:>7} {codes[1]:>8}")

print()
# A blanket failure makes "the two endpoints agree" true and meaningless, which is how
# a run from a host without bedrock:InvokeModel reads as a finding. Require at least
# one 200 before reporting agreement at all. Measured on a test host whose instance
# role lacked the permission: every cell above returned 403 and the agreement verdict
# fired regardless.
answered = [k for k, code in status.items() if code == 200]
split = [
    (m, t)
    for m in COVERED
    for t in TOOLSETS
    if (status[(m, t, "mantle")] == 200) != (status[(m, t, "runtime")] == 200)
]
if not answered:
    seen = sorted({code for code in status.values()})
    print(f"!! nothing was accepted: the only status codes seen were {seen}. Whatever")
    print("   they mean, no toolset support was measured here, and any claim that the")
    print("   endpoints agree would be vacuous. 403 means the caller cannot invoke")
    print("   these models at all -- fix that before reading anything below as a")
    print("   capability result.")
elif split:
    print(f"!! endpoint-dependent for {split}: read this as an ENDPOINT fact, not a")
    print("   model one, and do not reuse the per-model lists below across endpoints.")
else:
    print(f"=> {len(answered)} of {len(status)} (model, tool type, endpoint) calls "
          f"returned 200, and each")
    print("   (model, tool type) pair got the same answer from bedrock-mantle and from")
    print("   bedrock-runtime, so toolset support here is a property of the model")
    print("   rather than of the endpoint.")
for tool_type in TOOLSETS:
    yes = [m for m in COVERED if status[(m, tool_type, "mantle")] == 200]
    print(f"   {tool_type:28} accepted by {len(yes)}/{len(COVERED)}: {yes}")
for (model, endpoint), tools in sorted(offered.items()):
    print(f"   refused on {endpoint}: {model} offers instead {tools}")
if not offered:
    print("   No refusal quoted a per-model list in this run, so the second list this")
    print("   section describes is not visible here. Re-read the table above.")


model                        tool type                     mantle  runtime
---------------------------------------------------------------------------


anthropic.claude-haiku-4-5   computer_toolset_20260801        400      400


anthropic.claude-haiku-4-5   browser_toolset_20260801         400      400


anthropic.claude-opus-4-8    computer_toolset_20260801        200      200


anthropic.claude-opus-4-8    browser_toolset_20260801         200      200


anthropic.claude-opus-5      computer_toolset_20260801        200      200


anthropic.claude-opus-5      browser_toolset_20260801         200      200


anthropic.claude-sonnet-5    computer_toolset_20260801        200      200


anthropic.claude-sonnet-5    browser_toolset_20260801         200      200

=> 12 of 16 (model, tool type, endpoint) calls returned 200, and each
   (model, tool type) pair got the same answer from bedrock-mantle and from
   bedrock-runtime, so toolset support here is a property of the model
   rather than of the endpoint.
   computer_toolset_20260801    accepted by 3/4: ['anthropic.claude-opus-4-8', 'anthropic.claude-opus-5', 'anthropic.claude-sonnet-5']
   browser_toolset_20260801     accepted by 3/4: ['anthropic.claude-opus-4-8', 'anthropic.claude-opus-5', 'anthropic.claude-sonnet-5']
   refused on mantle: anthropic.claude-haiku-4-5 offers instead ['bash_20250124', 'computer_20250124', 'memory_20250818', 'text_editor_20250728', 'tool_search_tool_bm25_20251119', 'tool_search_tool_regex_20251119']
   refused on runtime: anthropic.claude-haiku-4-5 offers instead ['bash_20250124', 'computer_20250124', 'memory_20250818', 'text_editor_20250728', 'tool_search_tool_bm25_20251119', 'tool_sea

### The dispatch key, and §4's gate run against both shapes

One model, one prompt, two generations of the computer tool. The interesting line is not
the HTTP status — both are 200 — it is where the action ends up.


In [11]:
# Chosen from the measurement above rather than hard-coded, so this cell cannot outlive
# the support it depends on.
ga_models = [
    m for m in COVERED if status[(m, "computer_toolset_20260801", "mantle")] == 200
]
GA_MODEL = ga_models[0] if ga_models else None

shapes: dict[str, dict] = {}
if GA_MODEL is None:
    print("No model in COVERED accepts computer_toolset_20260801 on mantle today, so")
    print("there is nothing to compare. The table above is the finding in that case.")
else:
    arms = {
        "beta": ([computer_tool], {**AV, "anthropic-beta": COMPUTER_USE_BETA}),
        "toolset": ([{"type": "computer_toolset_20260801"}], dict(AV)),
    }
    for label, (tools, headers) in arms.items():
        code, data = post(
            f"{PREFIX}/messages",
            {
                "model": GA_MODEL,
                "max_tokens": 600,
                "messages": SCREENSHOT_ASK,
                "tools": tools,
            },
            region=REGION,
            headers=headers,
        )
        uses = tool_uses(data)
        print(f"{label:8} HTTP {code} | model {GA_MODEL} | stop_reason "
              f"{data.get('stop_reason')}")
        if code != 200 or not uses:
            print(f"         no tool_use block: {err(data)[:90]}")
            continue
        shapes[label] = uses[0]
        block = uses[0]
        print(f"         name={block['name']!r} "
              f"toolset_name={block.get('toolset_name')!r} "
              f"input={json.dumps(block['input'])}")

print()
if GA_MODEL is None:
    pass  # already reported above; there is nothing further to say
elif len(shapes) == 2:
    for label, block in shapes.items():
        # Section 4's code, unchanged, against each shape.
        outcome = perform_action(block["input"])
        detail = outcome.get("reason") or outcome.get("type") or "ok"
        state = "REFUSED" if outcome.get("refused") else "performed"
        print(f"{label:8} perform_action(input) -> {state}: {detail}")
    keys = {label: block["input"].get("action") for label, block in shapes.items()}
    print()
    print(f"=> input['action'] is {keys['beta']!r} under the beta type and "
          f"{keys['toolset']!r} under")
    print("   the toolset, so the gate in §4 cannot see the action at all in the second")
    print(f"   shape. There the action is the block name, "
          f"{shapes['toolset']['name']!r}, qualified")
    print(f"   by toolset_name={shapes['toolset'].get('toolset_name')!r}.")
else:
    print("Only one arm produced a tool_use block, so the comparison this cell exists")
    print("for did not happen. Re-run before drawing any conclusion from it.")


beta     HTTP 200 | model anthropic.claude-opus-4-8 | stop_reason tool_use
         name='computer' toolset_name=None input={"action": "screenshot"}


toolset  HTTP 200 | model anthropic.claude-opus-4-8 | stop_reason tool_use
         name='screenshot' toolset_name='computer' input={}

beta     perform_action(input) -> performed: image
toolset  perform_action(input) -> REFUSED: action 'None' requires human approval in this environment

=> input['action'] is 'screenshot' under the beta type and None under
   the toolset, so the gate in §4 cannot see the action at all in the second
   shape. There the action is the block name, 'screenshot', qualified
   by toolset_name='computer'.


### Two more request-shape rules

Both are measured, because both fail in a way that reads like something else: one as a
400 about a field you did copy correctly, the other as silence. The second is the reason
to read the `members` line below before keying a dispatch table on `name` alone.


In [12]:
if GA_MODEL is None:
    print("Skipped: no model in COVERED accepts the toolset on mantle today.")
else:
    code, data = post(
        f"{PREFIX}/messages",
        {
            "model": GA_MODEL,
            "max_tokens": 64,
            "messages": SCREENSHOT_ASK,
            "tools": [{"type": "computer_toolset_20260801", "name": "computer"}],
        },
        region=REGION,
        headers=dict(AV),
    )
    verdict = "accepted" if code == 200 else "rejected"
    print(f'toolset entry with "name" -> HTTP {code}, {verdict}')
    if code != 200:
        print(f"   {err(data, limit=300)}")

    # Each accepted toolset gets the ask suited to it, and we keep every tool_use block
    # in the reply rather than the first: the browser arm returns more than one.
    members: dict[str, set[str]] = {}
    for tool_type in TOOLSETS:
        if status[(GA_MODEL, tool_type, "mantle")] != 200:
            continue
        code, data = post(
            f"{PREFIX}/messages",
            {
                "model": GA_MODEL,
                "max_tokens": 600,
                "messages": ASKS[tool_type],
                "tools": [{"type": tool_type}],
            },
            region=REGION,
            headers=dict(AV),
        )
        for block in tool_uses(data):
            family = block.get("toolset_name") or tool_type
            members.setdefault(family, set()).add(block["name"])

    print()
    print("members returned in this run:", {k: sorted(v) for k, v in members.items()})
    if len(members) > 1:
        shared = sorted(set.intersection(*members.values()))
        if shared:
            print(f"=> {shared} came back from more than one toolset here, so a handler")
            print("   keyed on name alone is ambiguous. Dispatch on "
                  "(toolset_name, name).")
        else:
            print("=> no member name repeated across toolsets in this run. One run is")
            print("   not evidence of disjointness, so key on (toolset_name, name)")
            print("   anyway -- the field is there for this.")
    else:
        print("=> fewer than two toolsets answered, so this run cannot show whether")
        print("   member names collide. The (toolset_name, name) pair is still the")
        print("   key the response supplies for telling them apart.")


toolset entry with "name" -> HTTP 400, rejected
   tools.0.computer_toolset_20260801: name is not accepted on a toolset entry: the member names are fixed by the dated type, and the served tool keeps the family's bare name



members returned in this run: {'computer': ['screenshot'], 'browser': ['navigate', 'screenshot']}
=> ['screenshot'] came back from more than one toolset here, so a handler
   keyed on name alone is ambiguous. Dispatch on (toolset_name, name).


### Which list a refusal shows you, and why the middle one misleads

§4b's table above collects a refusal that quotes the model's own tool types. That is not
the only list the API hands back, and the difference matters when you are debugging a
400 rather than writing a table.

§1's second probe already produces one of these 400s and stops at the status code. Below
are three refusals of a tool `type` — same model, same prompt, all HTTP 400 — read for
the list each one offers. The middle one is the trap: a **missing beta header** is
reported as a type that does not exist, and the alternatives offered exclude the type you
asked for. Read that message as "this tool was removed" and you will go looking for a
replacement that is already in your request.


In [13]:
# Same model, same prompt, three tool "type" values. The point is which list comes back.
BETA_HDR = {**AV, "anthropic-beta": COMPUTER_USE_BETA}
EXPECTED = re.compile(r"expected tags: (.+)$")


def type_probe(tool: dict, headers: dict) -> tuple[int, dict]:
    return post(
        f"{PREFIX}/messages",
        {
            "model": GA_MODEL,
            # 64 is deliberate: this cell measures which refusal comes back, not
            # anything the model says, so a truncated reply costs nothing.
            "max_tokens": 64,
            "messages": [{"role": "user", "content": "hi"}],
            "tools": [tool],
        },
        region=REGION,
        headers=headers,
    )


if GA_MODEL is None:
    print("Skipped: nothing in COVERED accepted a toolset on mantle today.")
else:
    unknown_code, unknown_data = type_probe({"type": "banana"}, dict(AV))
    # computer_tool is §1's tool, unchanged, so the only variable below is the header.
    gated_code, gated_data = type_probe(computer_tool, dict(AV))
    allowed_code, allowed_data = type_probe(computer_tool, BETA_HDR)
    gated_type = computer_tool["type"]

    print(f"type 'banana', no beta header -> HTTP {unknown_code}: "
          f"{err(unknown_data, limit=200)}")
    print(f"type {gated_type!r}, no beta header -> HTTP {gated_code}")
    hit = EXPECTED.search(err(gated_data, limit=900))
    tags = [t.strip().strip("'") for t in hit.group(1).split(",")] if hit else []
    print(f"   alternatives offered ({len(tags)}): {tags}")
    print(f"type {gated_type!r}, WITH anthropic-beta {COMPUTER_USE_BETA} -> "
          f"HTTP {allowed_code}, stop_reason {allowed_data.get('stop_reason')}")

    print()
    if not tags:
        print("!! the second call quoted no list of alternatives, so this cell cannot")
        print(f"   show what it exists to show. Its message was: "
              f"{err(gated_data, limit=200)}")
    elif allowed_code != 200:
        print(f"!! the header did not make the request work ({allowed_code}), so the")
        print("   middle refusal cannot be attributed to the header. Nothing below")
        print("   would follow from this run.")
    else:
        if gated_type in tags:
            print(f"=> {gated_type} IS in the list it was refused against, so the")
            print("   trap this cell describes did not reproduce. Read the message.")
        else:
            print(f"=> {gated_type} is absent from the list its own refusal offers, yet")
            print("   the identical request with the header is 200. So that list is what")
            print("   the schema accepts given the headers you sent, not what the model")
            print("   supports.")
        inside = [t for t in TOOLSETS if t in tags]
        outside = [t for t in TOOLSETS if t not in tags]
        print(f"   of the two toolsets, {len(inside)} appear in it and {len(outside)} do"
              f" not: {inside or 'none'} vs {outside or 'none'}.")
        # The per-model list from the table above, for comparison. It is a different
        # kind of list, so a difference here is the finding, not a discrepancy to fix.
        per_model = sorted(set().union(*offered.values())) if offered else []
        only_per_model = [t for t in per_model if t not in tags]
        if not per_model:
            print("   No per-model list was collected above, so the two kinds of list")
            print("   cannot be compared in this run.")
        elif only_per_model:
            print(f"   {only_per_model} were offered by a model above but are not in this")
            print("   list, so the two are not the same question and neither is a")
            print("   superset of the other. Ask the model you are calling.")
        else:
            print("   Every type a model offered above also appears in this list in")
            print("   this run. That is one run, not a rule: the two lists answer")
            print("   different questions, so keep reading both.")


type 'banana', no beta header -> HTTP 400: tool type 'banana' is not supported for this model
type 'computer_20251124', no beta header -> HTTP 400
   alternatives offered (12): ['bash_20250124', 'browser_toolset_20260801', 'computer_toolset_20260801', 'custom', 'memory_20250818', 'text_editor_20250124', 'text_editor_20250429', 'text_editor_20250728', 'tool_search_tool_bm25', 'tool_search_tool_bm25_20251119', 'tool_search_tool_regex', 'tool_search_tool_regex_20251119']
type 'computer_20251124', WITH anthropic-beta computer-use-2025-11-24 -> HTTP 200, stop_reason max_tokens

=> computer_20251124 is absent from the list its own refusal offers, yet
   the identical request with the header is 200. So that list is what
   the schema accepts given the headers you sent, not what the model
   supports.
   of the two toolsets, 2 appear in it and 0 do not: ['computer_toolset_20260801', 'browser_toolset_20260801'] vs none.
   ['computer_20250124'] were offered by a model above but are not in this


## 5. Bash and text-editor tools

The same pattern with different tool types. These pair with the computer-use beta.

**The version suffix is part of the contract**, and a stale one is the most common
way this fails. `text_editor_20250124` is refused by the current models; the tool is
`text_editor_20250728`, and its `name` changed too — `str_replace_based_edit_tool`,
not `str_replace_editor`. The cell below sends both so you can see the refusal and
the fix side by side, and note that **the error lists the types the model does
accept**, which is the quickest way to find the current one.

In [10]:
bash_tool = {"type": "bash_20250124", "name": "bash"}
editor_stale = {"type": "text_editor_20250124", "name": "str_replace_editor"}
editor_current = {
    "type": "text_editor_20250728",
    "name": "str_replace_based_edit_tool",  # the name is version-specific too
}

for label, tool, prompt in [
    ("bash", bash_tool, "List the files in the current directory."),
    ("editor (stale type)", editor_stale, "Open /tmp/notes.txt and show its contents."),
    ("editor (current)", editor_current, "Open /tmp/notes.txt and show its contents."),
]:
    code, data = post(
        f"{PREFIX}/messages",
        {
            "model": OPUS48,
            "max_tokens": 500,
            "tools": [tool],
            "messages": [{"role": "user", "content": prompt}],
        },
        region=REGION,
        headers={**AV, "anthropic-beta": COMPUTER_USE_BETA},
    )
    print(f"{label:22} -> HTTP {code} | stop={data.get('stop_reason')}")
    for block in tool_uses(data):
        print(f"   {block['name']}: {json.dumps(block['input'])[:130]}")
    if code != 200:
        print(f"   {err(data)[:160]}")

print()
print("=> The refusal for the stale type lists the versions this model accepts.")
print("   Read it rather than guessing: these suffixes move with each beta.")

bash                   -> HTTP 200 | stop=tool_use
   bash: {"command": "ls -la"}


editor (stale type)    -> HTTP 400 | stop=None
   tool type 'text_editor_20250124' is not supported for this model


editor (current)       -> HTTP 200 | stop=tool_use
   str_replace_based_edit_tool: {"command": "view", "path": "/tmp/notes.txt"}

=> The refusal for the stale type lists the versions this model accepts.
   Read it rather than guessing: these suffixes move with each beta.


**Never execute these blindly.** A `bash` command or a file edit proposed by a
model is untrusted input. Allow-list what you will run, and sandbox it.

## 6. The memory tool

Gives Claude a directory it can read and write, so information survives beyond the
context window. Opt in with the **context-management** beta.

In [11]:
memory_tool = {"type": "memory_20250818", "name": "memory"}

code, data = post(
    f"{PREFIX}/messages",
    {
        "model": OPUS48,
        "max_tokens": 700,
        "tools": [memory_tool],
        "messages": [
            {
                "role": "user",
                "content": "Remember that our deploy tool is CodeDeploy and our "
                "primary Region is us-east-1.",
            }
        ],
    },
    region=REGION,
    headers={**AV, "anthropic-beta": CONTEXT_MGMT_BETA},
)
print("HTTP", code, "| stop_reason:", data.get("stop_reason"))
print("content block types:", [b.get("type") for b in data.get("content", [])])
for block in tool_uses(data):
    print(f"\nmemory command: {json.dumps(block['input'], indent=2)[:400]}")

HTTP 200 | stop_reason: tool_use
content block types: ['text', 'tool_use']

memory command: {
  "command": "view",
  "path": "/memories"
}


### A simulated memory backend

The model issues filesystem-style commands; you implement the store. Here it is
an in-memory dict — in production it would be a scoped directory, S3 prefix, or
database, isolated per user.

In [12]:
MEMORY_STORE: dict[str, str] = {}


def memory_backend(command: dict) -> dict:
    """Minimal memory implementation: view / create / str_replace / insert."""
    action = command.get("command")
    path = command.get("path", "")
    if action == "view":
        if path.rstrip("/") in ("", "/memories"):
            return {"entries": sorted(MEMORY_STORE)}
        return {
            "path": path,
            "content": MEMORY_STORE.get(path, ""),
            "exists": path in MEMORY_STORE,
        }
    if action == "create":
        MEMORY_STORE[path] = command.get("file_text", "")
        return {"ok": True, "path": path, "bytes": len(MEMORY_STORE[path])}
    if action == "str_replace":
        existing = MEMORY_STORE.get(path, "")
        MEMORY_STORE[path] = existing.replace(
            command.get("old_str", ""), command.get("new_str", "")
        )
        return {"ok": True, "path": path}
    if action == "insert":
        MEMORY_STORE[path] = MEMORY_STORE.get(path, "") + command.get("insert_line", "")
        return {"ok": True, "path": path}
    return {"error": f"unsupported command {action}"}


def memory_session(prompt: str, model: str = OPUS48, max_rounds: int = 5) -> str:
    conversation = [{"role": "user", "content": prompt}]
    for _ in range(max_rounds):
        code, data = post(
            f"{PREFIX}/messages",
            {
                "model": model,
                "max_tokens": 900,
                "tools": [memory_tool],
                "messages": conversation,
            },
            region=REGION,
            headers={**AV, "anthropic-beta": CONTEXT_MGMT_BETA},
        )
        if code != 200:
            return f"HTTP {code}: {err(data)[:120]}"
        blocks = tool_uses(data)
        if not blocks:
            return claude_text(data)
        conversation.append({"role": "assistant", "content": data["content"]})
        results = []
        for block in blocks:
            outcome = memory_backend(block["input"])
            print(
                f"   memory: {block['input'].get('command'):12} "
                f"{block['input'].get('path', ''):28} -> {json.dumps(outcome)[:70]}"
            )
            results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block["id"],
                    "content": json.dumps(outcome),
                }
            )
        conversation.append({"role": "user", "content": results})
    return "(max rounds reached)"


print("--- session 1: store facts ---")
print(
    "reply:",
    memory_session(
        "Remember: our deploy tool is CodeDeploy, primary Region us-east-1. "
        "Save it to memory, then confirm briefly."
    )[:200],
)
print("\nstore now holds:", {k: v[:60] for k, v in MEMORY_STORE.items()})

--- session 1: store facts ---


   memory: view         /memories                    -> {"entries": []}


   memory: create       /memories/deployment_info.md -> {"ok": true, "path": "/memories/deployment_info.md", "bytes": 81}


reply: Saved. Deploy tool is **CodeDeploy**, primary Region **us-east-1**.

store now holds: {'/memories/deployment_info.md': '# Deployment Info\n\n- **Deploy tool:** CodeDeploy\n- **Primary'}


In [13]:
print("--- session 2: fresh conversation, recall from memory ---")
print(
    "reply:",
    memory_session("Check your memory directory and tell me which deploy tool we use.")[
        :250
    ],
)

--- session 2: fresh conversation, recall from memory ---


   memory: view         /memories                    -> {"entries": ["/memories/deployment_info.md"]}


   memory: view         /memories/deployment_info.md -> {"path": "/memories/deployment_info.md", "content": "# Deployment Info


reply: According to my memory directory, we use **AWS CodeDeploy** as our deploy tool. The primary region noted is **us-east-1**.


The second session shares **no conversation history** with the first — only the
memory store. That is the point: state outlives the context window.

## 7. Context management (compaction)

Long tool-using agents accumulate enormous tool-result history. `context_management`
lets Claude clear old tool calls so the context stays affordable.

**Two things have to be right, and getting either wrong makes it a silent no-op.**

1. **You must supply a `trigger`.** `{"edits": [{"type": "clear_tool_uses_20250919"}]}`
   on its own is accepted and does nothing — no error, no clearing. Earlier versions
   of this notebook sent exactly that and were demonstrating a no-op. Give it
   `trigger: {"type": "input_tokens", "value": N}`, and optionally
   `keep: {"type": "tool_uses", "value": N}` to retain the most recent few.
2. **Read `context_management.applied_edits` on the response.** That is the
   authoritative signal, and it reports `cleared_input_tokens` and
   `cleared_tool_uses`. Inferring from token counts alone cannot distinguish "did not
   fire" from "fired and found nothing".

The cell below sends **one fixed transcript** of synthetic tool history at three
sizes, twice each — once with a trigger and once without — so any difference is
attributable to compaction alone. Running the agent twice would not work as a
control: the model chooses how many tools to call, so the two runs diverge.

In [14]:
code, data = post(
    f"{PREFIX}/messages",
    {
        "model": OPUS48,
        "max_tokens": 300,
        "messages": [{"role": "user", "content": "Reply OK"}],
        "context_management": {"edits": [{"type": "clear_tool_uses_20250919"}]},
    },
    region=REGION,
    headers={**AV, "anthropic-beta": CONTEXT_MGMT_BETA},
)
print("context_management ->", code, "|", claude_text(data)[:60] or err(data)[:80])

context_management -> 200 | OK


In [15]:
bulky_tool = {
    "name": "fetch_log",
    "description": "Fetch a log extract.",
    "input_schema": {
        "type": "object",
        "properties": {"service": {"type": "string"}},
        "required": ["service"],
    },
}


def fetch_log(service: str) -> dict:
    # Deliberately verbose, like a real log tool.
    return {
        "service": service,
        "lines": [
            f"{service} INFO handled request {i} in {10 + i}ms" for i in range(60)
        ],
    }


def transcript(rounds: int) -> list:
    """A synthetic agent history: `rounds` turns of bulky tool results."""
    services = ["api", "worker", "scheduler", "gateway", "cache", "queue"]
    history = [
        {"role": "user", "content": "Check the logs, then tell me which is slowest."}
    ]
    for n, service in enumerate(services[:rounds]):
        call_id = f"toolu_synthetic_{n:02d}"
        history.append(
            {
                "role": "assistant",
                "content": [
                    {"type": "tool_use", "id": call_id, "name": "fetch_log",
                     "input": {"service": service}}
                ],
            }
        )
        history.append(
            {
                "role": "user",
                "content": [
                    {"type": "tool_result", "tool_use_id": call_id,
                     "content": json.dumps(fetch_log(service))}
                ],
            }
        )
    history.append({"role": "user", "content": "Which service is slowest?"})
    return history


# No trigger vs an explicit one. The first is the shape most examples show, and it
# does nothing.
NO_TRIGGER = {"edits": [{"type": "clear_tool_uses_20250919"}]}
WITH_TRIGGER = {
    "edits": [
        {
            "type": "clear_tool_uses_20250919",
            "trigger": {"type": "input_tokens", "value": 2000},
            "keep": {"type": "tool_uses", "value": 1},  # retain the most recent
        }
    ]
}


def ask(history, context_management=None):
    body = {
        "model": OPUS48,
        "max_tokens": 300,
        "tools": [bulky_tool],
        "messages": history,
    }
    if context_management:
        body["context_management"] = context_management
    code, data = post(
        f"{PREFIX}/messages", body, region=REGION,
        headers={**AV, "anthropic-beta": CONTEXT_MGMT_BETA},
    )
    if code != 200:
        return None, f"HTTP {code}"
    edits = (data.get("context_management") or {}).get("applied_edits") or []
    return (data.get("usage") or {}).get("input_tokens"), edits


print(f"{'rounds':>7} {'no edit':>9} {'no trigger':>11} {'with trigger':>13}  applied_edits")
print("-" * 84)
for rounds in (2, 4, 6):
    history = transcript(rounds)
    plain, _ = ask(history)
    untriggered, no_edits = ask(history, NO_TRIGGER)
    triggered, edits = ask(history, WITH_TRIGGER)
    detail = (
        f"cleared {edits[0].get('cleared_input_tokens')} tok / "
        f"{edits[0].get('cleared_tool_uses')} tool uses" if edits else "none"
    )
    print(f"{rounds:>7} {plain!s:>9} {untriggered!s:>11} {triggered!s:>13}  {detail}")
    if not no_edits:
        print(f"{'':>7} {'':>9} {'^ no trigger => applied_edits is [] : nothing happened'}")

 rounds   no edit  no trigger  with trigger  applied_edits
------------------------------------------------------------------------------------


      2      2632        2632          1611  cleared 986 tok / 1 tool uses
                  ^ no trigger => applied_edits is [] : nothing happened


      4      5118        5118          1889  cleared 3174 tok / 3 tool uses
                  ^ no trigger => applied_edits is [] : nothing happened


      6      7418        7418          1964  cleared 5384 tok / 5 tool uses
                  ^ no trigger => applied_edits is [] : nothing happened


Read the middle column against the right-hand one. Without a `trigger`,
`applied_edits` comes back `[]` and the input-token count is identical to sending no
`context_management` at all — the parameter is accepted and inert. With a trigger,
`applied_edits` names exactly what went: at six rounds of history it cleared over
5,000 input tokens and five tool uses, holding the payload near the trigger value
however long the conversation gets.

That flat ceiling is the point. Combine it with prompt caching (see
`02-thinking-tools-and-caching.ipynb`) and a long agent run stays affordable instead
of growing without bound.

Two cautions. `keep` decides how much recent context the model still has to work
with, so setting it too low will make the agent forget what it just did — tune it
against your task, not for the token saving. And a cleared tool result is *gone from
the model's view*: if your agent needs to refer back to something, put it in the
memory tool (§6), not in the tool history.

## 8. All three together: a long-running agent shape

Memory for durable facts, compaction for transient tool noise, caching for the
static instructions.

In [16]:
AGENT_SYSTEM = (
    "You are an operations assistant. Use the memory tool to persist "
    "durable facts about the environment. Use fetch_log for logs. "
    "Be concise."
) * 40  # padded so caching is worthwhile

code, data = post(
    f"{PREFIX}/messages",
    {
        "model": OPUS48,
        "max_tokens": 900,
        # Static instructions, cached with a 1-hour TTL.
        "system": [
            {
                "type": "text",
                "text": AGENT_SYSTEM,
                "cache_control": {"type": "ephemeral", "ttl": "1h"},
            }
        ],
        "tools": [memory_tool, bulky_tool],
        "context_management": {"edits": [{"type": "clear_tool_uses_20250919"}]},
        "messages": [
            {
                "role": "user",
                "content": "What deploy tool do we use? Check memory first.",
            }
        ],
    },
    region=REGION,
    headers={**AV, "anthropic-beta": CONTEXT_MGMT_BETA},
)
usage = data.get("usage", {})
print("HTTP", code)
print(
    "cache_write:",
    usage.get("cache_creation_input_tokens"),
    "| cache_read:",
    usage.get("cache_read_input_tokens"),
)
print("blocks:", [b.get("type") for b in data.get("content", [])])
for block in tool_uses(data):
    print(f"   {block['name']}: {json.dumps(block['input'])[:110]}")

HTTP 200
cache_write: 3493 | cache_read: 0
blocks: ['text', 'tool_use']
   memory: {"command": "view", "path": "/memories"}


## 9. Safety checklist before you ship any of this

| Control | Why |
|---|---|
| Dedicated VM / container, least privilege | Contains mistakes and attacks |
| No sensitive credentials in reach | Anything visible can be exfiltrated |
| Domain allow-list for network access | Limits exposure to malicious content |
| Action allow-list in your executor | The model proposes; you decide |
| Human approval for consequential actions | Payments, deletions, consent clicks |
| Per-user memory isolation | One user's memory must never leak into another's |
| Beta tool type versions | The suffix and the `name` both move per beta; a stale pair is a 400 that lists the valid types |
| `clear_tool_uses` without a `trigger` | Accepted and **inert**. `applied_edits` comes back `[]`. Always set a trigger and read `applied_edits` (§7) |
| Audit log of every executed action | You will need it |
| Inform users and obtain consent | Required, and the right thing to do |

In [17]:
# The gate from §4, restated as the pattern to copy.
DANGEROUS = {"left_click", "type", "key", "middle_click", "double_click", "scroll"}


def gated_executor(action: dict, human_approves=lambda a: False) -> dict:
    kind = action.get("action")
    if kind in DANGEROUS and not human_approves(action):
        return {"refused": True, "reason": "awaiting human approval"}
    return perform_action(action)


print("screenshot   ->", json.dumps(gated_executor({"action": "screenshot"}))[:70])
print(
    "left_click   ->",
    json.dumps(gated_executor({"action": "left_click", "coordinate": [100, 200]})),
)
print(
    "left_click ✓ ->",
    json.dumps(gated_executor({"action": "left_click"}, human_approves=lambda a: True)),
)

screenshot   -> {"type": "image", "source": {"type": "base64", "media_type": "image/jp
left_click   -> {"refused": true, "reason": "awaiting human approval"}
left_click ✓ -> {"refused": true, "reason": "action 'left_click' requires human approval in this environment"}


## Gotchas — Claude agentic tools on bedrock-mantle

| Gotcha | Detail |
|---|---|
| Beta opt-in | `anthropic-beta` **header** on mantle (body field on runtime) — needed by the dated types in §1, not by the toolsets in §4b |
| Version pairing | A beta tool `type` must match its beta version, e.g. `computer_20251124`; a `*_toolset_20260801` type pairs with nothing |
| Model support | Not every Claude model takes every tool type, and the two generations differ — probe both, §2 and §4b |
| Dispatch key | Beta types put the action in `input['action']`; toolsets put it in the block `name` with a `toolset_name` — §4b |
| Three refusals | A `type` the service does not know gets no list; a beta type sent without its header gets the schema's tags, which exclude it; a type this model does not take gets that model's own list — §4b |
| Memory beta | Uses `context-management-2025-06-27`, not the computer-use beta |
| You implement the backend | Memory and GUI actions do nothing unless your code acts |
| Untrusted proposals | `bash` / editor / click actions are untrusted input — allow-list |
| Memory isolation | Scope the store per user, or you leak across tenants |
| Compaction + caching | Combine `clear_tool_uses` with cached system prompts |
| Beta Service terms | Computer use is Beta under the AWS Service Terms |

## Where next
- `01-messages-api-core.ipynb` · `02-thinking-tools-and-caching.ipynb`
- Server-side tools on another family:
  `../01-openai-gpt/05-server-side-tools-and-fine-tuning.ipynb`

## Converse in earnest — the tool loop, provider parameters, and caching

The earlier endpoint section proved this model answers through Converse. That is
the easy part. This section does the three things you actually need on
`bedrock-runtime`, because each differs from the `bedrock-mantle` equivalent:

1. **A complete tool round trip** — `toolUse` out, `toolResult` back in. Getting a
   tool *call* is half the job; feeding the result back is where the shapes bite.
2. **`additionalModelRequestFields`** — Converse normalises the common fields, so
   anything provider-specific goes through this escape hatch.
3. **`cachePoint`** — prompt caching is a first-class Converse block, and support
   for it is per model rather than universal.


In [18]:
from bedrock import converse_text, converse_tool_uses, resolve_runtime_id, runtime_client

RUNTIME_ID = "anthropic.claude-sonnet-5"
runtime = runtime_client(REGION)
resolved = resolve_runtime_id(RUNTIME_ID, REGION)

# Converse tool shape: toolSpec, and the JSON Schema nests under inputSchema.json.
# This is NOT the OpenAI shape - there is no {"type": "function"} wrapper.
WEATHER_TOOL = {
    "toolSpec": {
        "name": "get_weather",
        "description": "Current weather for a city",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            }
        },
    }
}

history = [
    {"role": "user", "content": [{"text": "What is the weather in Singapore? Use the tool."}]}
]
first = runtime.converse(
    modelId=resolved,
    messages=history,
    toolConfig={"tools": [WEATHER_TOOL]},
    inferenceConfig={"maxTokens": 500},
)
print("turn 1 stop reason:", first.get("stopReason"))
print("turn 1 blocks     :", [next(iter(b)) for b in first["output"]["message"]["content"]])

uses = converse_tool_uses(first)
if not uses:
    print("no tool call this run - tool_choice defaults to the model's discretion;")
    print("retry, or set toolConfig['toolChoice'] to compel one.")
else:
    use = uses[0]
    print(f"tool call         : {use['name']}({use['input']})")
    # Validate before acting on it. A malformed call still reports tool_use.
    city = str(use["input"].get("city", "")).lower()
    print("arguments valid   :", "yes" if "singapore" in city else f"NO ({use['input']})")

    # Echo the assistant turn back VERBATIM, then answer with a toolResult whose
    # toolUseId matches. Dropping either breaks the loop with a 400.
    history.append(first["output"]["message"])
    history.append(
        {
            "role": "user",
            "content": [
                {
                    "toolResult": {
                        "toolUseId": use["toolUseId"],
                        "content": [{"json": {"tempC": 31, "conditions": "humid"}}],
                    }
                }
            ],
        }
    )
    second = runtime.converse(
        modelId=resolved,
        messages=history,
        toolConfig={"tools": [WEATHER_TOOL]},
        inferenceConfig={"maxTokens": 300},
    )
    print("turn 2 stop reason:", second.get("stopReason"))
    print("final answer      :", converse_text(second).strip()[:160])


turn 1 stop reason: tool_use
turn 1 blocks     : ['reasoningContent', 'toolUse']
tool call         : get_weather({'city': 'Singapore'})
arguments valid   : yes


turn 2 stop reason: end_turn
final answer      : The current weather in Singapore is **humid** with a temperature of **31°C**.


In [19]:
# Converse normalises maxTokens, temperature, topP and stopSequences. Anything
# provider-specific goes through additionalModelRequestFields, unvalidated by
# Converse and passed to the provider as-is. That makes it powerful and sharp:
# a key this model does not recognise is a 400, not a silent no-op.
from bedrock import converse_reasoning

PUZZLE = (
    "A bat and ball cost $1.10 together. The bat costs $1.00 more than the ball. "
    "How much is the ball?"
)

for label, extra in [
    ("no extra fields", None),
    ("provider fields", {"thinking": {"type": "adaptive"}}),
]:
    kwargs = {"additionalModelRequestFields": extra} if extra else {}
    try:
        response = runtime.converse(
            modelId=resolved,
            messages=[{"role": "user", "content": [{"text": PUZZLE}]}],
            inferenceConfig={"maxTokens": 900},
            **kwargs,
        )
    except Exception as exc:
        print(f"{label:<16} {type(exc).__name__}: {str(exc)[-90:]}")
        continue
    blocks = [next(iter(b)) for b in response["output"]["message"]["content"]]
    trace = converse_reasoning(response)
    answer = converse_text(response).strip().replace("\n", " ")
    print(f"{label:<16} out={response['usage']['outputTokens']:>4} blocks={blocks}")
    print(f"{'':<16} reasoning={len(trace)} chars | {answer[:70]}")

print()
print("Note whether a reasoningContent block appears above. Some models return the")
print("trace as a typed block on Converse and some do not, so read the blocks")
print("rather than assuming - and never index content[0].")


no extra fields  out= 256 blocks=['text']
                 reasoning=0 chars | # Bat and Ball Problem  **The ball costs $0.05 (5 cents)**  ## Quick c


provider fields  out= 300 blocks=['reasoningContent', 'text']
                 reasoning=0 chars | **The ball costs $0.05 (5 cents).**  Here's the reasoning:  - Let the 

Note whether a reasoningContent block appears above. Some models return the
trace as a typed block on Converse and some do not, so read the blocks
rather than assuming - and never index content[0].


In [20]:
# cachePoint marks a prefix as cacheable. Everything BEFORE the marker is cached;
# the marker goes last in the block list it applies to. Watch the usage fields:
# the first call writes, the second reads.
HANDBOOK = "You are a support handbook. " + (
    "Retries: use exponential backoff with full jitter, cap at 16 seconds. " * 160
)


def cached_call():
    return runtime.converse(
        modelId=resolved,
        system=[{"text": HANDBOOK}, {"cachePoint": {"type": "default"}}],
        messages=[{"role": "user", "content": [{"text": "One line: the retry policy?"}]}],
        inferenceConfig={"maxTokens": 60},
    )


print(f"{'call':<6} {'write':>8} {'read':>8}  total")
print("-" * 40)
for n in (1, 2):
    usage = cached_call()["usage"]
    print(
        f"{n:<6} {str(usage.get('cacheWriteInputTokens')):>8} "
        f"{str(usage.get('cacheReadInputTokens')):>8}  {usage['totalTokens']}"
    )

print()
print("Call 1 writes the prefix, call 2 reads it back. Cached input is billed at a")
print("lower rate than fresh input, so a long stable system prompt reused across")
print("many calls is where this pays. Check the pricing page for the current ratio.")


call      write     read  total
----------------------------------------


1          None     4332  4376


2          None     4332  4370

Call 1 writes the prefix, call 2 reads it back. Cached input is billed at a
lower rate than fresh input, so a long stable system prompt reused across
many calls is where this pays. Check the pricing page for the current ratio.


### What this section adds over the endpoint check above

- **The tool loop is the part that bites.** `toolSpec` is not the OpenAI shape,
  the JSON Schema nests under `inputSchema.json`, and the second turn must echo the
  assistant message back verbatim alongside a `toolResult` whose `toolUseId`
  matches. Miss any of that and you get a 400.
- **`additionalModelRequestFields` is unvalidated by Converse.** It is the only way
  to reach provider-specific behaviour, and a key the model does not recognise
  fails the call rather than being ignored.
- **Feature support is per model, not per endpoint.** Read the output above rather
  than carrying an assumption over from another family.
